# External Valuation
## March 7, 2026


**Target variable**

`sport_type_grouped`

**External dataset**

`data/data_for_modeling_Mar4.csv`

**Model**

XGBoost pipeline loaded from `model_pkls/xgboost_pipeline.pkl`

**Label handling**

- Unseen label `EBikeRide` mapped to `Ride` before evaluation
- Label encoding reconstructed with training classes: `Hike` / `Ride` / `Run` / `Walk` / `Workout`

**Feature engineering applied to external data**
- `has_gps` ← 1 if `num_turns` is not null, else 0
- GPS shape features (`num_turns`, `turns_per_mile`, `wobble`, `sprawl`) ← NaN filled with 0
- `is_winter` ← 1 if month ∈ {12, 1, 2}
- `is_summer` ← 1 if month ∈ {6, 7, 8}

**Evaluation metrics**

- Accuracy 
- Macro F1 
- Classification Report 
- Confusion Matrix 
- Top misclassification pairs


In [4]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, log_loss

In [6]:
warnings.filterwarnings("ignore", category=UserWarning)

In [7]:
model = joblib.load("model_pkls/xgboost_pipeline.pkl")

In [8]:
# Feature list must match what was used during training exactly
feature_cols = [
    # Core workout intensity / size
    "speed_mph", "miles", "moving_time", "elapsed_time",
    "moving_time_per", "feet_per_mile",
    # Route / GPS-shape features (+ indicator)
    "has_gps", "num_turns", "turns_per_mile", "wobble", "sprawl",
    # Time patterns
    "hour", "month", "dayofweek", "is_weekend",
    # Context flags
    "manual", "is_winter", "is_summer",
    # Categorical (one-hot encoded inside pipeline)
    "day_part",
]

target_col = "sport_type_grouped"

In [9]:
# Reconstruct the label encoder with the same classes seen during training
le = LabelEncoder()
le.classes_ = pd.Index(["Hike", "Ride", "Run", "Walk", "Workout"])

In [10]:
df = pd.read_csv("data/data_for_modeling_Mar4.csv")

In [11]:
# Keep only rows with a known target label
df = df[df[target_col].notna()].copy()

In [15]:
# Map unseen labels to the closest known class
df[target_col] = df[target_col].replace({"EBikeRide": "Ride"})

In [16]:
# Boolean flag for whether GPS data was available
df["has_gps"] = df["num_turns"].notna().astype(int)

# Fill missing GPS-shape features with 0
gps_cols     = ["num_turns", "turns_per_mile", "wobble", "sprawl"]
df[gps_cols] = df[gps_cols].fillna(0)

# Seasonal flags — use existing month column if available
if "month" in df.columns:
    month = df["month"]
else:
    df["start_date_local"] = pd.to_datetime(df["start_date_local"], errors="coerce")
    month = df["start_date_local"].dt.month

df["is_winter"] = month.isin([12, 1, 2]).astype(int)
df["is_summer"] = month.isin([6, 7, 8]).astype(int)

In [17]:
# Encode target
y = le.transform(df[target_col])

In [18]:
# Align features to training schema
missing_cols = [c for c in feature_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"External data is missing required columns: {missing_cols}")

X = df[feature_cols].copy()

In [19]:
def evaluate_model(name, model):
    print(f"\n--- {name} Performance ---")

    y_pred = model.predict(X)

    print(f"External Accuracy : {accuracy_score(y, y_pred):.4f}")
    print(f"External Macro F1 : {f1_score(y, y_pred, average='macro'):.4f}")

    print("\nClassification Report:")
    print(classification_report(y, y_pred, target_names=le.classes_))

    # Confusion matrix
    cm    = confusion_matrix(y, y_pred, labels=list(range(len(le.classes_))))
    cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
    print("\nConfusion Matrix:")
    print(cm_df)

    # Top misclassification pairs
    results            = df[[target_col]].copy()
    results["y_true"]  = df[target_col].values          
    results["y_pred"]  = le.inverse_transform(y_pred)
    results["correct"] = results["y_true"] == results["y_pred"]

    top_errors = (
        results[~results["correct"]]
        .groupby(["y_true", "y_pred"])
        .size()
        .sort_values(ascending=False)
        .head(15)
    )
    print("\nTop misclassification pairs:")
    print(top_errors)

In [20]:
evaluate_model("XGBoost", model)


--- XGBoost Performance ---
External Accuracy : 0.8886
External Macro F1 : 0.7844

Classification Report:
              precision    recall  f1-score   support

        Hike       0.38      0.39      0.38        49
        Ride       0.94      0.90      0.92       730
         Run       0.89      0.95      0.92       860
        Walk       0.85      0.83      0.84       342
     Workout       0.89      0.83      0.86       334

    accuracy                           0.89      2315
   macro avg       0.79      0.78      0.78      2315
weighted avg       0.89      0.89      0.89      2315


Confusion Matrix:
         Hike  Ride  Run  Walk  Workout
Hike       19     1    1    28        0
Ride        0   658   45     1       26
Run         4    21  820    12        3
Walk       22     4   26   284        6
Workout     5    14   30     9      276

Top misclassification pairs:
y_true   y_pred 
Ride     Run        45
Workout  Run        30
Hike     Walk       28
Ride     Workout    26
Walk  

XGBoost — Test vs. External Evaluation
<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: #f5f5f5; border-bottom: 2px solid #ccc;">
      <th style="text-align: left; padding: 10px 16px;">Dataset</th>
      <th style="text-align: center; padding: 10px 16px;">Accuracy</th>
      <th style="text-align: center; padding: 10px 16px;">Macro F1</th>
      <th style="text-align: center; padding: 10px 16px;">Hike F1 (Minority Class)</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #ebf3fb; border-bottom: 1px solid #e8e8e8;">
      <td style="text-align: left; padding: 9px 16px;"><strong>Test (Internal)</strong></td>
      <td style="text-align: center; padding: 9px 16px;"><strong>0.9179</strong></td>
      <td style="text-align: center; padding: 9px 16px;"><strong>0.8441</strong></td>
      <td style="text-align: center; padding: 9px 16px;"><strong>0.63</strong></td>
    </tr>
    <tr style="background-color: #ffffff; border-bottom: 1px solid #e8e8e8;">
      <td style="text-align: left; padding: 9px 16px;">External Evaluation</td>
      <td style="text-align: center; padding: 9px 16px;">0.8886</td>
      <td style="text-align: center; padding: 9px 16px;">0.7844</td>
      <td style="text-align: center; padding: 9px 16px;">0.38</td>
    </tr>
  </tbody>
</table>